# Notebook 05 – Integração com Oracle Cloud Object Storage

Este notebook tem como objetivo integrar os artefatos do projeto (datasets e modelos) armazenados no Google Drive ao Oracle Cloud Infrastructure (OCI) Object Storage. Ele centraliza a autenticação, upload, download e validação dos arquivos, servindo como pipeline de publicação entre a etapa de Ciência de Dados e o backend da aplicação.

Instalando o SDK no Colab

In [10]:
!pip install oci

Montagem do Drive

In [11]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Variáveis para não repetir o caminho

In [12]:
BASE_PATH = "/content/drive/MyDrive/Hackathon_OCI_G9/HACKATHON G9 - TEAM 20"

DATASETS_PATH = f"{BASE_PATH}/datasets"
MODELOS_PATH = f"{BASE_PATH}/modelos"

Verificar se as pastas existem

In [13]:
import os

print(os.path.exists(DATASETS_PATH))
print(os.path.exists(MODELOS_PATH))

True
True


Verificar o conteúdo

In [14]:
for arquivo in os.listdir(DATASETS_PATH):
    print(arquivo)

sidra_pof_gastos_alimentares.csv
perfis_financeiros_sinteticos.csv
transacoes_sinteticas.csv
bcb_inadimplencia_pf.csv
sidra_despesas_limpo.csv
sidra_despesas_bruto.csv
sidra_despesas_por_categoria_renda.csv


In [15]:
for arquivo in os.listdir(MODELOS_PATH):
    print(arquivo)

modelo_perfil_producao.pkl
escalador_perfil.pkl
codificador_perfil.pkl
modelo_perfil_deep_learning.h5
tokenizador_categoria.pkl
modelo_categoria_deep_learning.h5
modelo_categoria_producao.pkl
codificador_categorias.pkl
vetorizador_tfidf.pkl


Criando o arquivo config

In [16]:
config_content = """
[DEFAULT]
user=ocid1.user.oc1..aaaaaaaakonhg3qhcs5gxasc4pcinmpbnydhb7odflbkn6te37jvoxo5daya
fingerprint=89:87:d7:57:ac:b3:3c:66:11:9b:63:28:b7:6c:70:dd
tenancy=ocid1.tenancy.oc1..aaaaaaaam6lrs4by2xgob66ap7guu3otaq4vbsltl5uwxrrr6wa2z4kruaya
region=sa-saopaulo-1
key_file=/content/jpdeodato@gmail.com-2026-07-20T16_25_43.660Z.pem
"""

with open("/content/config", "w") as f:
    f.write(config_content)

print("Arquivo config criado com sucesso!")

Arquivo config criado com sucesso!


Testando a autenticação

In [17]:
import oci

config = oci.config.from_file("/content/config")

object_storage = oci.object_storage.ObjectStorageClient(config)

namespace = object_storage.get_namespace().data

print("Namespace:", namespace)


Namespace: grrjqrgpt6r9


Listando os buckets

In [18]:
config = oci.config.from_file("/content/config")

object_storage = oci.object_storage.ObjectStorageClient(config)

namespace = object_storage.get_namespace().data
compartment_id = config["tenancy"]

buckets = object_storage.list_buckets(
    namespace_name=namespace,
    compartment_id=compartment_id
)

for bucket in buckets.data:
    print(bucket.name)

hackathon-one-g9-team-20


Função para fazer upload arquivo por arquivo

In [19]:
def upload_file(local_path, object_name):

    with open(local_path, "rb") as arquivo:

        object_storage.put_object(
            namespace_name=namespace,
            bucket_name=bucket_name,
            object_name=object_name,
            put_object_body=arquivo
        )

    print(f"✔ {object_name}")

Enviando todos os datasets

In [21]:
bucket_name = "hackathon-one-g9-team-20"

for arquivo in os.listdir(DATASETS_PATH):

    caminho = os.path.join(DATASETS_PATH, arquivo)

    if os.path.isfile(caminho):

        upload_file(
            caminho,
            f"datasets/{arquivo}"
        )

✔ datasets/sidra_pof_gastos_alimentares.csv
✔ datasets/perfis_financeiros_sinteticos.csv
✔ datasets/transacoes_sinteticas.csv
✔ datasets/bcb_inadimplencia_pf.csv
✔ datasets/sidra_despesas_limpo.csv
✔ datasets/sidra_despesas_bruto.csv
✔ datasets/sidra_despesas_por_categoria_renda.csv


Enviando todos os modelos

In [22]:
for arquivo in os.listdir(MODELOS_PATH):

    caminho = os.path.join(MODELOS_PATH, arquivo)

    if os.path.isfile(caminho):

        upload_file(
            caminho,
            f"models/{arquivo}"
        )

✔ models/modelo_perfil_producao.pkl
✔ models/escalador_perfil.pkl
✔ models/codificador_perfil.pkl
✔ models/modelo_perfil_deep_learning.h5
✔ models/tokenizador_categoria.pkl
✔ models/modelo_categoria_deep_learning.h5
✔ models/modelo_categoria_producao.pkl
✔ models/codificador_categorias.pkl
✔ models/vetorizador_tfidf.pkl


Conferindo o Bucket

In [23]:
objects = object_storage.list_objects(
    namespace_name=namespace,
    bucket_name=bucket_name
)

for obj in objects.data.objects:
    print(obj.name)

datasets/bcb_inadimplencia_pf.csv
datasets/perfis_financeiros_sinteticos.csv
datasets/sidra_despesas_bruto.csv
datasets/sidra_despesas_limpo.csv
datasets/sidra_despesas_por_categoria_renda.csv
datasets/sidra_pof_gastos_alimentares.csv
datasets/transacoes_sinteticas.csv
models/codificador_categorias.pkl
models/codificador_perfil.pkl
models/escalador_perfil.pkl
models/modelo_categoria_deep_learning.h5
models/modelo_categoria_producao.pkl
models/modelo_perfil_deep_learning.h5
models/modelo_perfil_producao.pkl
models/tokenizador_categoria.pkl
models/vetorizador_tfidf.pkl
